# Market Mix Modeling with PyMC

# 0. Libs and Source directory

In [1]:
import os

notebook_path = os.path.abspath("")
source_folder = "src"
source_path = notebook_path[:notebook_path.find(source_folder) + len(source_folder)]

os.chdir(source_path)

In [2]:
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pymc_extras.prior import Prior

from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.transformers import geometric_adstock, logistic_saturation

warnings.filterwarnings("ignore", category=FutureWarning)

az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = [12, 7]
plt.rcParams["figure.dpi"] = 100

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

# 1. Data

In [100]:
dt_prophet_holidays = pd.read_csv("data/03_primary/dt_prophet_holidays.csv")
dt_simulated_weekly = pd.read_csv("data/03_primary/dt_simulated_weekly.csv")

In [101]:
# Parameters
window_start = "2016-01-01"
window_end = "2018-12-31"
country = "DE"

In [124]:
dt_simulated_weekly = dt_simulated_weekly.rename({"DATE":"ds"}, axis=1)
df_sales = dt_simulated_weekly[dt_simulated_weekly["ds"].between(window_start, window_end)].copy()
df_sales["ds"] = pd.to_datetime(df_sales["ds"], format="%Y-%m-%d")

df_sales["year"] = pd.to_datetime(df_sales["ds"]).dt.year
df_sales["month"] = pd.to_datetime(df_sales["ds"]).dt.month

In [125]:
df_holidays = dt_prophet_holidays[dt_prophet_holidays["ds"].between(window_start, window_end) & (dt_prophet_holidays["country"] == country)].copy()
df_holidays["ds"] = pd.to_datetime(df_holidays["ds"], format="%Y-%m-%d")
df_holidays["month"] = pd.to_datetime(df_holidays["ds"]).dt.month

df_holidays = df_holidays.groupby(["year", "month"],as_index=False).agg(holidays=("holiday","count"))

In [126]:
df_sales = df_sales.merge(df_holidays, on=["year", "month"], how="left").fillna(0)
df_sales["events"] = np.where(df_sales["events"] == "na", 0, 1)

df_sales = df_sales.drop(["year", "month"], axis=1)

In [128]:
# Parameters from robyn optimization
model_spec = {
    "facebook_S": {
        "alpha": 2.992003,
        "gamma": 0.949203,
        "theta": 0.017215,
    },
    "newsletter": {
        "alpha": 1.102317,
        "gamma": 0.998069,
        "theta": 0.390837,
    },
    "ooh_S": {
        "alpha": 0.500313,
        "gamma": 0.348616,
        "theta": 0.394762,
    },
    "print_S": {
        "alpha": 2.950387,
        "gamma": 0.314979,
        "theta": 0.366256,
    },
    "search_S": {
        "alpha": 0.718514,
        "gamma": 0.458433,
        "theta": 0.11901,
    },
    "tv_S": {
        "alpha": 2.916362,
        "gamma": 0.612891,
        "theta": 0.526186,
    },
}